# Per-Dataset Appendix Metrics

Self-contained notebook for appendix tables with datasets kept separate. It loads the final benchmark parquet files directly and computes one row per dataset and method.

In [ ]:
from __future__ import annotations

from pathlib import Path
import re
import sys
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

NOTEBOOK_DIR = Path.cwd().resolve()
ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
RESULTS_DIR = ROOT / 'results'

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)
warnings.filterwarnings('ignore', category=FutureWarning)

DATASET_ORDER = [
    'adult',
    'compas',
    'german_credit',
    'give_me_some_credit',
    'heloc',
    'lending_club',
    'wisconsin_breast_cancer',
]

PAPER_CERTCF_ALPHA = 0.20
PAPER_CERTCF_LAMBDA = 1.0
PAPER_CERTCF_LABEL = f'CertCF a={PAPER_CERTCF_ALPHA:.2f} lam={PAPER_CERTCF_LAMBDA:g}'
BASELINE_METHOD_ORDER = ['NN10000', 'Growing Spheres', 'DiCE', 'FACE']
METHOD_ORDER = [PAPER_CERTCF_LABEL, *BASELINE_METHOD_ORDER]
METHOD_DISPLAY = {
    PAPER_CERTCF_LABEL: 'CertCF',
    'NN10000': 'NN',
    'Growing Spheres': 'Growing Spheres',
    'DiCE': 'DiCE',
    'FACE': 'FACE',
}

ROBUSTNESS_SIGMA_GRID = [0.00, 0.01, 0.03, 0.05, 0.10]
ROBUSTNESS_CAT_FLIP_GRID = [0.00, 0.01, 0.03, 0.05, 0.10]
ROBUSTNESS_GRID_N_PERTURBATIONS = 10
ROBUSTNESS_BATCH_SIZE = 8192
ROBUSTNESS_RANDOM_STATE = 0
FINAL_BENCHMARK_CONFIG = ROOT / 'configs' / 'benchmarks' / 'final_benchmark.yaml'

L0_TOL = 1e-6
LOF_N_NEIGHBORS = 20
SURROGATE_TREE_MAX_DEPTH = 3
SURROGATE_TREE_MIN_SAMPLES_LEAF = 5
SURROGATE_TREE_TEST_SIZE = 0.30
SURROGATE_TREE_RANDOM_STATE = 0
SURROGATE_TREE_MIN_SAMPLES = 20
EXPORT_TABLES = False
EXPORT_DIR = NOTEBOOK_DIR / 'data' / 'generated' if NOTEBOOK_DIR.name == 'notebooks' else ROOT / 'notebooks' / 'data' / 'generated'


## Load Benchmark Results

The quality metrics use the final paper benchmark files. DiCE's batch-size-one result file is loaded separately and used only to override the query-time column.

In [ ]:
CERTCF_SWEEP_RE = re.compile(r'eps_alpha=([0-9.]+)-sparsity_lambda=([0-9.]+)')
CERTCF_OLD_RE = re.compile(r'eps_alpha=([0-9.]+)')


def parse_certcf_hparams(run_name: str) -> tuple[float, float]:
    run_name = str(run_name)
    match = CERTCF_SWEEP_RE.search(run_name)
    if match:
        return float(match.group(1)), float(match.group(2))
    match = CERTCF_OLD_RE.search(run_name)
    if match:
        return float(match.group(1)), np.nan
    return np.nan, np.nan


def method_label(row: pd.Series) -> str:
    method = str(row['method'])
    run_name = str(row.get('run_name', method))
    if method == 'certcf':
        alpha, lam = parse_certcf_hparams(run_name)
        if np.isfinite(alpha) and np.isfinite(lam):
            return f'CertCF a={alpha:.2f} lam={lam:g}'
        if np.isfinite(alpha):
            return f'CertCF old a={alpha:.2f}'
        return run_name
    if method == 'nearest_neighbor':
        return 'NN10000'
    if method == 'growing_spheres':
        return 'Growing Spheres'
    if method == 'dice':
        return 'DiCE'
    if method == 'face':
        return 'FACE'
    return run_name


RESULT_FILE_SPECS = [
    {
        'result_key': 'final_benchmark_noface',
        'path': RESULTS_DIR / 'final_benchmark_noface.parquet',
        'excluded_methods': {'face', 'certcf'},
        'included_run_names_by_method': {'growing_spheres': {'gs_max_radius=50.0'}},
    },
    {
        'result_key': 'final_benchmark_noface_certcf_shrink_sparsity',
        'path': RESULTS_DIR / 'final_benchmark_noface_certcf_shrink_sparsity.parquet',
        'included_methods': {'certcf'},
    },
    {
        'result_key': 'final_benchmark_face',
        'path': RESULTS_DIR / 'final_benchmark_face.parquet',
        'included_methods': {'face'},
    },
]

DICE_BATCH1_QUERY_TIME_PATH = RESULTS_DIR / 'dice_query_batch1_20queries.parquet'

file_status = pd.DataFrame([
    {
        'result_key': spec['result_key'],
        'path': str(spec['path']),
        'available': spec['path'].exists(),
        'size_mb': spec['path'].stat().st_size / (1024 ** 2) if spec['path'].exists() else np.nan,
    }
    for spec in RESULT_FILE_SPECS
] + [{
    'result_key': 'dice_batch1_query_time',
    'path': str(DICE_BATCH1_QUERY_TIME_PATH),
    'available': DICE_BATCH1_QUERY_TIME_PATH.exists(),
    'size_mb': DICE_BATCH1_QUERY_TIME_PATH.stat().st_size / (1024 ** 2) if DICE_BATCH1_QUERY_TIME_PATH.exists() else np.nan,
}])
display(file_status)

missing = file_status.loc[~file_status['available'], 'path'].tolist()
if missing:
    raise FileNotFoundError('Missing result files:\n' + '\n'.join(missing))

frames = []
for spec in RESULT_FILE_SPECS:
    frame = pd.read_parquet(spec['path']).copy()
    included_methods = spec.get('included_methods')
    if included_methods is not None:
        frame = frame[frame['method'].astype(str).isin(included_methods)].copy()
    excluded_methods = spec.get('excluded_methods', set())
    if excluded_methods:
        frame = frame[~frame['method'].astype(str).isin(excluded_methods)].copy()
    for method_name, run_names in spec.get('included_run_names_by_method', {}).items():
        method_mask = frame['method'].astype(str).eq(str(method_name))
        frame = frame[~method_mask | frame['run_name'].astype(str).isin(run_names)].copy()
    if 'run_name' not in frame.columns:
        frame['run_name'] = frame['method'].astype(str)
    frame['result_key'] = spec['result_key']
    frame['result_file'] = spec['path'].name
    frames.append(frame)

RAW_BENCHMARK_DF = pd.concat(frames, ignore_index=True)
RAW_BENCHMARK_DF['method_label'] = RAW_BENCHMARK_DF.apply(method_label, axis=1)
parsed = RAW_BENCHMARK_DF['run_name'].map(parse_certcf_hparams)
RAW_BENCHMARK_DF['certcf_alpha'] = [x[0] for x in parsed]
RAW_BENCHMARK_DF['certcf_sparsity_lambda'] = [x[1] for x in parsed]

BENCHMARK_DF = RAW_BENCHMARK_DF[RAW_BENCHMARK_DF['method_label'].astype(str).isin(METHOD_ORDER)].copy()
BENCHMARK_DF['dataset'] = pd.Categorical(BENCHMARK_DF['dataset'], DATASET_ORDER, ordered=True)
BENCHMARK_DF['method_label'] = pd.Categorical(BENCHMARK_DF['method_label'].astype(str), METHOD_ORDER, ordered=True)
BENCHMARK_DF = BENCHMARK_DF.sort_values(['dataset', 'method_label', 'query_idx'], ignore_index=True)

if PAPER_CERTCF_LABEL not in set(BENCHMARK_DF['method_label'].astype(str)):
    available = sorted(RAW_BENCHMARK_DF.loc[RAW_BENCHMARK_DF['method'].astype(str).eq('certcf'), 'method_label'].astype(str).unique())
    raise ValueError(f'Missing selected CertCF run {PAPER_CERTCF_LABEL}. Available CertCF labels: {available}')

DICE_BATCH1_DF = pd.read_parquet(DICE_BATCH1_QUERY_TIME_PATH).copy()
DICE_BATCH1_DF['method_label'] = DICE_BATCH1_DF.apply(method_label, axis=1)
DICE_BATCH1_DF = DICE_BATCH1_DF[DICE_BATCH1_DF['method_label'].astype(str).eq('DiCE')].copy()

print({
    'rows': int(len(BENCHMARK_DF)),
    'datasets': int(BENCHMARK_DF['dataset'].nunique()),
    'methods': BENCHMARK_DF['method_label'].dropna().astype(str).unique().tolist(),
    'dice_batch1_rows': int(len(DICE_BATCH1_DF)),
})


## Shared Helpers

In [ ]:
from counterfactuals.benchmarks import create_default_registries
from sklearn.neighbors import NearestNeighbors


def indexed_columns(df: pd.DataFrame, prefix: str) -> list[str]:
    cols = [c for c in df.columns if c.startswith(prefix)]
    return sorted(cols, key=lambda c: int(c.rsplit('_', 1)[1]))


def x_cf_columns_for_dim(dim: int) -> list[str]:
    return [f'x_cf_{idx}' for idx in range(int(dim))]


def method_display(label: str) -> str:
    return METHOD_DISPLAY.get(str(label), str(label))


def add_method_display(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out['method'] = out['method_label'].astype(str).map(method_display)
    out['dataset'] = pd.Categorical(out['dataset'].astype(str), DATASET_ORDER, ordered=True)
    out['method'] = pd.Categorical(out['method'].astype(str), [method_display(m) for m in METHOD_ORDER], ordered=True)
    return out.sort_values(['dataset', 'method'], ignore_index=True)


def load_train_xy_by_dataset(dataset_order: list[str]) -> dict[str, tuple[np.ndarray, np.ndarray]]:
    registries = create_default_registries()
    train_by_dataset = {}
    for dataset_name in dataset_order:
        dataset = registries['dataset'].create(dataset_name, data_dir=str(ROOT / 'data'))
        dataset.load()
        x_train, y_train = dataset.get_train()
        train_by_dataset[dataset_name] = (
            np.asarray(x_train, dtype=np.float32),
            np.asarray(y_train),
        )
    return train_by_dataset


def nn10000_l1_scale_by_dataset(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for dataset in DATASET_ORDER:
        ds = df[df['dataset'].astype(str).eq(dataset)]
        nn = ds[
            ds['method_label'].astype(str).eq('NN10000')
            & ds['success'].astype(bool)
            & ds['l1_distance'].notna()
        ]
        fallback = ds[ds['success'].astype(bool) & ds['l1_distance'].notna()]
        if not nn.empty:
            scale = float(nn['l1_distance'].mean())
            source = 'NN10000 mean successful L1'
        elif not fallback.empty:
            scale = float(fallback['l1_distance'].median())
            source = 'all-method median successful L1 fallback'
        else:
            scale = np.nan
            source = 'missing'
        rows.append({'dataset': dataset, 'l1_scale': scale, 'scale_source': source})
    return pd.DataFrame(rows)


## Basic Metrics

Validity, L1 proximity, L0 sparsity, and query time are read directly from the benchmark rows. DiCE query time is overridden with the batch-size-one timing file.

In [ ]:
def compute_validity(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for dataset_name in DATASET_ORDER:
        ds = df[df['dataset'].astype(str).eq(dataset_name)]
        for method_label in METHOD_ORDER:
            method_df = ds[ds['method_label'].astype(str).eq(method_label)]
            if method_df.empty:
                continue
            success = method_df['success'].astype(bool)
            rows.append({
                'dataset': dataset_name,
                'method_label': method_label,
                'n_queries': int(len(method_df)),
                'n_success': int(success.sum()),
                'validity_ratio': float(success.mean()),
            })
    return pd.DataFrame(rows)


def compute_l1_proximity(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for dataset_name in DATASET_ORDER:
        ds = df[df['dataset'].astype(str).eq(dataset_name)]
        for method_label in METHOD_ORDER:
            method_df = ds[
                ds['method_label'].astype(str).eq(method_label)
                & ds['success'].astype(bool)
                & ds['l1_distance'].notna()
            ]
            if method_df.empty:
                continue
            rows.append({
                'dataset': dataset_name,
                'method_label': method_label,
                'n_l1_success': int(len(method_df)),
                'l1_proximity': float(method_df['l1_distance'].mean()),
            })
    return pd.DataFrame(rows)


def compute_l0_sparsity(df: pd.DataFrame, tol: float = L0_TOL) -> pd.DataFrame:
    rows = []
    for dataset_name in DATASET_ORDER:
        ds = df[df['dataset'].astype(str).eq(dataset_name)]
        if ds.empty:
            continue
        orig_cols = [c for c in indexed_columns(ds, 'x_orig_') if ds[c].notna().any()]
        cf_cols = [f"x_cf_{c.rsplit('_', 1)[1]}" for c in orig_cols]
        if not orig_cols or not set(cf_cols).issubset(ds.columns):
            continue
        for method_label in METHOD_ORDER:
            method_df = ds[
                ds['method_label'].astype(str).eq(method_label)
                & ds['success'].astype(bool)
            ]
            if method_df.empty:
                continue
            x_orig = method_df[orig_cols].to_numpy(dtype=float)
            x_cf = method_df[cf_cols].to_numpy(dtype=float)
            finite = np.isfinite(x_orig).all(axis=1) & np.isfinite(x_cf).all(axis=1)
            if not finite.any():
                continue
            l0 = (np.abs(x_cf[finite] - x_orig[finite]) > tol).sum(axis=1)
            rows.append({
                'dataset': dataset_name,
                'method_label': method_label,
                'n_l0_success': int(len(l0)),
                'l0_sparsity': float(np.mean(l0)),
            })
    return pd.DataFrame(rows)


def compute_query_time(df: pd.DataFrame, dice_batch1_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for dataset_name in DATASET_ORDER:
        ds = df[df['dataset'].astype(str).eq(dataset_name)]
        for method_label in METHOD_ORDER:
            source_df = ds[ds['method_label'].astype(str).eq(method_label)]
            source = 'final_benchmark'
            if method_label == 'DiCE':
                dice_ds = dice_batch1_df[dice_batch1_df['dataset'].astype(str).eq(dataset_name)]
                if not dice_ds.empty:
                    source_df = dice_ds
                    source = 'dice_batch1_20queries'
            if source_df.empty or 'runtime_s' not in source_df.columns:
                continue
            runtime = pd.to_numeric(source_df['runtime_s'], errors='coerce').dropna().to_numpy(dtype=float)
            if len(runtime) == 0:
                continue
            rows.append({
                'dataset': dataset_name,
                'method_label': method_label,
                'n_query_time_rows': int(len(runtime)),
                'query_time_s': float(np.mean(runtime)),
                'query_time_source': source,
            })
    return pd.DataFrame(rows)


VALIDITY_DF = compute_validity(BENCHMARK_DF)
L1_PROXIMITY_DF = compute_l1_proximity(BENCHMARK_DF)
SPARSITY_DF = compute_l0_sparsity(BENCHMARK_DF)
QUERY_TIME_DF = compute_query_time(BENCHMARK_DF, DICE_BATCH1_DF)

for name, frame in [('validity', VALIDITY_DF), ('l1', L1_PROXIMITY_DF), ('l0', SPARSITY_DF), ('query_time', QUERY_TIME_DF)]:
    print(name, frame.shape)


## Empirical Robustness

This recomputes the post-hoc perturbation grid and then averages the 25 grid cells per dataset and method.

In [ ]:
import yaml
from dataset_specs import get_tabular_dataset_spec
from scripts.benchmark import _build_torch_model_from_checkpoint


def load_robustness_models(config_path: Path = FINAL_BENCHMARK_CONFIG) -> dict[str, object]:
    with open(config_path, 'r') as f:
        cfg = yaml.safe_load(f)
    models = {}
    for ds_cfg in cfg['datasets']:
        dataset_name = ds_cfg['name']
        model_params = dict(ds_cfg['model']['params'])
        checkpoint = ROOT / model_params['checkpoint']
        models[dataset_name] = _build_torch_model_from_checkpoint(
            checkpoint=str(checkpoint),
            device='cpu',
            dataset_module=model_params.get('dataset_module', dataset_name),
            hidden_dims=list(model_params.get('hidden_dims', [64, 32])),
            dropout=float(model_params.get('dropout', 0.2)),
        )
    return models


def predict_in_batches(model, x: np.ndarray, batch_size: int = ROBUSTNESS_BATCH_SIZE) -> np.ndarray:
    preds = []
    for start in range(0, len(x), batch_size):
        preds.append(model.predict(x[start:start + batch_size].astype(np.float32, copy=False)))
    return np.concatenate(preds)


def snap_ohe_blocks(x: np.ndarray, categorical_slices: tuple[tuple[int, int], ...]) -> np.ndarray:
    out = x.copy()
    for start, end in categorical_slices:
        if end <= start:
            continue
        chosen = np.argmax(out[:, start:end], axis=1)
        out[:, start:end] = 0.0
        out[np.arange(len(out)), start + chosen] = 1.0
    return out


def perturb_counterfactuals(x_cf: np.ndarray, spec, n_perturbations: int, numerical_sigma: float, categorical_flip_p: float, rng: np.random.Generator) -> np.ndarray:
    numerical_dims = [
        start
        for feature_type, (start, end) in zip(tuple(spec.input_types), tuple(spec.feature_slices))
        if feature_type == 'numerical'
    ]
    categorical_slices = tuple(spec.categorical_slices)
    base = snap_ohe_blocks(x_cf, categorical_slices)
    perturbed = np.repeat(base, n_perturbations, axis=0)

    if numerical_dims and numerical_sigma > 0:
        perturbed[:, numerical_dims] += rng.normal(0.0, numerical_sigma, size=(len(perturbed), len(numerical_dims)))

    if categorical_flip_p > 0:
        for start, end in categorical_slices:
            width = end - start
            if width <= 1:
                continue
            current = np.argmax(perturbed[:, start:end], axis=1)
            flip = rng.random(len(perturbed)) < categorical_flip_p
            if not flip.any():
                continue
            replacement = rng.integers(0, width - 1, size=int(flip.sum()))
            replacement = replacement + (replacement >= current[flip])
            perturbed[flip, start:end] = 0.0
            perturbed[np.flatnonzero(flip), start + replacement] = 1.0

    return perturbed.astype(np.float32, copy=False)


def compute_robustness_grid(df: pd.DataFrame) -> pd.DataFrame:
    models = load_robustness_models()
    rows = []
    for sigma_idx, sigma in enumerate(ROBUSTNESS_SIGMA_GRID):
        for flip_idx, categorical_flip_p in enumerate(ROBUSTNESS_CAT_FLIP_GRID):
            for dataset_idx, dataset_name in enumerate(DATASET_ORDER):
                spec = get_tabular_dataset_spec(dataset_name)
                model = models[dataset_name]
                cf_cols = x_cf_columns_for_dim(spec.n_features)
                ds = df[df['dataset'].astype(str).eq(dataset_name)]
                if ds.empty or not set(cf_cols).issubset(ds.columns):
                    continue
                for method_idx, method_label in enumerate(METHOD_ORDER):
                    method_df = ds[
                        ds['method_label'].astype(str).eq(method_label)
                        & ds['success'].astype(bool)
                    ].copy()
                    if method_df.empty:
                        continue
                    seed = ROBUSTNESS_RANDOM_STATE + 100000 * sigma_idx + 10000 * flip_idx + 1000 * dataset_idx + 100 * method_idx
                    rng = np.random.default_rng(seed)
                    x_cf = method_df[cf_cols].to_numpy(dtype=np.float32)
                    finite = np.isfinite(x_cf).all(axis=1)
                    if not finite.any():
                        continue
                    method_df = method_df.loc[finite]
                    x_cf = x_cf[finite]
                    targets = method_df['y_cf'].fillna(method_df['target_class']).to_numpy(dtype=int)
                    x_perturbed = perturb_counterfactuals(
                        x_cf=x_cf,
                        spec=spec,
                        n_perturbations=ROBUSTNESS_GRID_N_PERTURBATIONS,
                        numerical_sigma=sigma,
                        categorical_flip_p=categorical_flip_p,
                        rng=rng,
                    )
                    pred = predict_in_batches(model, x_perturbed)
                    stable = pred == np.repeat(targets, ROBUSTNESS_GRID_N_PERTURBATIONS)
                    rows.append({
                        'dataset': dataset_name,
                        'method_label': method_label,
                        'numerical_sigma': float(sigma),
                        'categorical_flip_p': float(categorical_flip_p),
                        'n_success': int(len(x_cf)),
                        'n_perturbed': int(len(x_perturbed)),
                        'target_robustness_pct': 100.0 * float(np.mean(stable)),
                    })
    return pd.DataFrame(rows)


def summarize_robustness_grid(grid_df: pd.DataFrame) -> pd.DataFrame:
    return (
        grid_df
        .groupby(['dataset', 'method_label'], observed=True)
        .agg(
            robustness_grid_cells=('target_robustness_pct', 'size'),
            robustness_n_success=('n_success', 'max'),
            robustness_avg=('target_robustness_pct', 'mean'),
        )
        .reset_index()
    )


ROBUSTNESS_GRID_DF = compute_robustness_grid(BENCHMARK_DF)
ROBUSTNESS_DF = summarize_robustness_grid(ROBUSTNESS_GRID_DF)
print('robustness', ROBUSTNESS_DF.shape)


## Plausibility And Privacy

Plausibility is mean pointwise `log10(LOF)`. The relative proximity ratio compares CF-to-train same-class NN distance against natural train-to-train same-class NN distance.

In [ ]:
from sklearn.neighbors import LocalOutlierFactor


def compute_log10_lof(df: pd.DataFrame) -> pd.DataFrame:
    train_by_dataset = load_train_xy_by_dataset(DATASET_ORDER)
    rows = []
    for dataset_name in DATASET_ORDER:
        x_train, _ = train_by_dataset[dataset_name]
        cf_cols = x_cf_columns_for_dim(x_train.shape[1])
        ds = df[df['dataset'].astype(str).eq(dataset_name)]
        if ds.empty or not set(cf_cols).issubset(ds.columns):
            continue
        lof = LocalOutlierFactor(n_neighbors=min(LOF_N_NEIGHBORS, max(1, len(x_train) - 1)), novelty=True, n_jobs=-1)
        lof.fit(x_train)
        for method_label in METHOD_ORDER:
            method_df = ds[
                ds['method_label'].astype(str).eq(method_label)
                & ds['success'].astype(bool)
            ]
            if method_df.empty:
                continue
            x_cf = method_df[cf_cols].to_numpy(dtype=np.float32)
            finite = np.isfinite(x_cf).all(axis=1)
            if not finite.any():
                continue
            positive_lof = np.maximum(-lof.score_samples(x_cf[finite]), 1e-12)
            rows.append({
                'dataset': dataset_name,
                'method_label': method_label,
                'n_lof_success': int(len(positive_lof)),
                'log10_lof': float(np.mean(np.log10(positive_lof))),
                'positive_lof_mean': float(np.mean(positive_lof)),
            })
    return pd.DataFrame(rows)


def training_same_class_closest_nn_distance_distributions(x_train: np.ndarray, y_train: np.ndarray) -> dict[int, np.ndarray]:
    y_train = np.asarray(y_train, dtype=int).reshape(-1)
    distributions = {}
    for label in np.unique(y_train):
        class_x = x_train[y_train == int(label)]
        if len(class_x) <= 1:
            continue
        nn = NearestNeighbors(n_neighbors=2, metric='manhattan', n_jobs=-1)
        nn.fit(class_x)
        distances, _ = nn.kneighbors(class_x, return_distance=True)
        distributions[int(label)] = np.sort(distances[:, 1].astype(float))
    return distributions


def same_class_closest_nn_distances(x_cf: np.ndarray, labels: np.ndarray, x_train: np.ndarray, y_train: np.ndarray, reference_distributions: dict[int, np.ndarray]) -> np.ndarray:
    labels = np.asarray(labels, dtype=int).reshape(-1)
    y_train = np.asarray(y_train, dtype=int).reshape(-1)
    nn_l1 = np.full(len(x_cf), np.nan, dtype=float)
    for label in np.unique(labels):
        train_mask = y_train == int(label)
        cf_mask = labels == int(label)
        reference = reference_distributions.get(int(label))
        if reference is None or train_mask.sum() < 1 or not cf_mask.any():
            continue
        nn = NearestNeighbors(n_neighbors=1, metric='manhattan', n_jobs=-1)
        nn.fit(x_train[train_mask])
        distances, _ = nn.kneighbors(x_cf[cf_mask], return_distance=True)
        nn_l1[np.flatnonzero(cf_mask)] = distances[:, 0].astype(float)
    return nn_l1


def compute_relative_proximity_ratio(df: pd.DataFrame) -> pd.DataFrame:
    train_by_dataset = load_train_xy_by_dataset(DATASET_ORDER)
    rows = []
    for dataset_name in DATASET_ORDER:
        x_train, y_train = train_by_dataset[dataset_name]
        reference_distributions = training_same_class_closest_nn_distance_distributions(x_train, y_train)
        train_distances = np.concatenate(list(reference_distributions.values())) if reference_distributions else np.array([], dtype=float)
        train_mean = float(np.mean(train_distances)) if len(train_distances) else np.nan
        cf_cols = x_cf_columns_for_dim(x_train.shape[1])
        ds = df[df['dataset'].astype(str).eq(dataset_name)]
        if ds.empty or not set(cf_cols).issubset(ds.columns):
            continue
        for method_label in METHOD_ORDER:
            method_df = ds[
                ds['method_label'].astype(str).eq(method_label)
                & ds['success'].astype(bool)
            ]
            if method_df.empty:
                continue
            x_cf = method_df[cf_cols].to_numpy(dtype=np.float32)
            finite = np.isfinite(x_cf).all(axis=1)
            if not finite.any():
                continue
            method_df = method_df.loc[finite]
            x_cf = x_cf[finite]
            labels = method_df['y_cf'].fillna(method_df['target_class']).to_numpy(dtype=int)
            cf_distances = same_class_closest_nn_distances(x_cf, labels, x_train, y_train, reference_distributions)
            cf_distances = cf_distances[np.isfinite(cf_distances)]
            if len(cf_distances) == 0 or not np.isfinite(train_mean) or train_mean <= 0:
                continue
            cf_mean = float(np.mean(cf_distances))
            rows.append({
                'dataset': dataset_name,
                'method_label': method_label,
                'n_relative_proximity_success': int(len(cf_distances)),
                'cf_same_class_nn_l1': cf_mean,
                'train_same_class_nn_l1': train_mean,
                'relative_proximity_ratio': float(cf_mean / train_mean),
            })
    return pd.DataFrame(rows)


LOF_DF = compute_log10_lof(BENCHMARK_DF)
RELATIVE_PROXIMITY_DF = compute_relative_proximity_ratio(BENCHMARK_DF)
print('lof', LOF_DF.shape)
print('relative proximity', RELATIVE_PROXIMITY_DF.shape)


## Decision Tree Interpretability

In [ ]:
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor


def compute_tree_surrogate_error(df: pd.DataFrame) -> pd.DataFrame:
    scale_by_dataset = nn10000_l1_scale_by_dataset(df).set_index('dataset')['l1_scale'].to_dict()
    rows = []
    for dataset_name in DATASET_ORDER:
        ds = df[df['dataset'].astype(str).eq(dataset_name)]
        if ds.empty:
            continue
        orig_cols = [c for c in indexed_columns(ds, 'x_orig_') if ds[c].notna().any()]
        cf_cols = [f"x_cf_{c.rsplit('_', 1)[1]}" for c in orig_cols]
        if not orig_cols or not set(cf_cols).issubset(ds.columns):
            continue
        scale = scale_by_dataset.get(dataset_name, np.nan)
        if not np.isfinite(scale) or scale <= 0:
            scale = 1.0
        for method_label in METHOD_ORDER:
            method_df = ds[
                ds['method_label'].astype(str).eq(method_label)
                & ds['success'].astype(bool)
            ]
            if len(method_df) < SURROGATE_TREE_MIN_SAMPLES:
                continue
            x_query = method_df[orig_cols].to_numpy(dtype=float)
            x_cf = method_df[cf_cols].to_numpy(dtype=float)
            finite = np.isfinite(x_query).all(axis=1) & np.isfinite(x_cf).all(axis=1)
            x_query = x_query[finite]
            x_cf = x_cf[finite]
            if len(x_query) < SURROGATE_TREE_MIN_SAMPLES:
                continue
            train_idx, test_idx = train_test_split(np.arange(len(x_query)), test_size=SURROGATE_TREE_TEST_SIZE, random_state=SURROGATE_TREE_RANDOM_STATE)
            tree = DecisionTreeRegressor(max_depth=SURROGATE_TREE_MAX_DEPTH, min_samples_leaf=SURROGATE_TREE_MIN_SAMPLES_LEAF, random_state=SURROGATE_TREE_RANDOM_STATE)
            tree.fit(x_query[train_idx], x_cf[train_idx])
            pred = tree.predict(x_query[test_idx])
            true = x_cf[test_idx]
            l1_error = np.abs(pred - true).sum(axis=1)
            rows.append({
                'dataset': dataset_name,
                'method_label': method_label,
                'n_tree_success': int(len(x_query)),
                'tree_n_test': int(len(test_idx)),
                'tree_r2': float(r2_score(true, pred, multioutput='variance_weighted')),
                'decision_tree_error': float(np.mean(l1_error / scale)),
                'decision_tree_raw_l1_error': float(np.mean(l1_error)),
            })
    return pd.DataFrame(rows)


TREE_ERROR_DF = compute_tree_surrogate_error(BENCHMARK_DF)
print('tree', TREE_ERROR_DF.shape)


## Final Tables

In [ ]:
required_metric_frames = [
    'VALIDITY_DF',
    'L1_PROXIMITY_DF',
    'SPARSITY_DF',
    'ROBUSTNESS_DF',
    'TREE_ERROR_DF',
    'QUERY_TIME_DF',
    'LOF_DF',
    'RELATIVE_PROXIMITY_DF',
]

missing_metric_frames = [name for name in required_metric_frames if name not in globals()]
if missing_metric_frames:
    missing = ', '.join(missing_metric_frames)
    raise RuntimeError(
        f'Missing metric dataframe(s): {missing}. Run the metric computation cells above '
        'this section first. LOF_DF and RELATIVE_PROXIMITY_DF are created in the '
        '"Plausibility And Privacy" cell.'
    )

metric_frames = [globals()[name] for name in required_metric_frames]

PER_DATASET_METRICS_DF = metric_frames[0].copy()
for frame in metric_frames[1:]:
    keep_cols = ['dataset', 'method_label'] + [
        col for col in frame.columns
        if col not in {'dataset', 'method_label'} and not col.startswith('n_')
    ]
    PER_DATASET_METRICS_DF = PER_DATASET_METRICS_DF.merge(frame[keep_cols], on=['dataset', 'method_label'], how='outer')

PER_DATASET_METRICS_DF = add_method_display(PER_DATASET_METRICS_DF)
metric_cols = [
    'validity_ratio',
    'l1_proximity',
    'l0_sparsity',
    'robustness_avg',
    'decision_tree_error',
    'query_time_s',
    'log10_lof',
    'relative_proximity_ratio',
]
PER_DATASET_METRICS_DF = PER_DATASET_METRICS_DF[['dataset', 'method', *metric_cols]]

print(PER_DATASET_METRICS_DF.shape)
display(PER_DATASET_METRICS_DF.round({
    'validity_ratio': 4,
    'l1_proximity': 4,
    'l0_sparsity': 4,
    'robustness_avg': 2,
    'decision_tree_error': 4,
    'query_time_s': 4,
    'log10_lof': 4,
    'relative_proximity_ratio': 4,
}))

METHOD_TABLES = {
    str(method): (
        PER_DATASET_METRICS_DF[PER_DATASET_METRICS_DF['method'].astype(str).eq(str(method))]
        .drop(columns='method')
        .reset_index(drop=True)
    )
    for method in [method_display(m) for m in METHOD_ORDER]
}

for method, table in METHOD_TABLES.items():
    print(f'\n{method}')
    display(table.round(4))


## LaTeX Tables By Dataset

Generate one compact LaTeX table per dataset, with methods as rows and the appendix metrics as columns.

In [ ]:
DATASET_DISPLAY = {
    'adult': 'Adult',
    'compas': 'COMPAS',
    'german_credit': 'German Credit',
    'give_me_some_credit': 'Give Me Some Credit',
    'heloc': 'HELOC',
    'lending_club': 'Lending Club',
    'wisconsin_breast_cancer': 'Wisconsin Breast Cancer',
}

LATEX_DATASET_LABEL = {
    'adult': 'adult',
    'compas': 'compas',
    'german_credit': 'german-credit',
    'give_me_some_credit': 'give-me-some-credit',
    'heloc': 'heloc',
    'lending_club': 'lending-club',
    'wisconsin_breast_cancer': 'wisconsin-breast-cancer',
}

LATEX_METRIC_SPECS = [
    ('Method', 'method', None),
    ('Validity $\\uparrow$', 'validity_ratio', '{:.3f}'),
    ('$L_1$ $\\downarrow$', 'l1_proximity', '{:.3f}'),
    ('$L_0$ $\\downarrow$', 'l0_sparsity', '{:.2f}'),
    ('Robust. $\\uparrow$', 'robustness_avg', '{:.1f}'),
    ('DT err. $\\downarrow$', 'decision_tree_error', '{:.3f}'),
    ('Query (s) $\\downarrow$', 'query_time_s', '{:.3f}'),
    ('$\\log_{10}(\\mathrm{LOF})$ $\\downarrow$', 'log10_lof', '{:.3f}'),
    ('RPR $\\to 1$', 'relative_proximity_ratio', '{:.3f}'),
]


def latex_metric_value(value: object, fmt: str | None) -> str:
    if fmt is None:
        return str(value)
    value = pd.to_numeric(pd.Series([value]), errors='coerce').iloc[0]
    if not np.isfinite(value):
        return '--'
    return fmt.format(float(value))


def dataset_metrics_to_latex(dataset_name: str) -> str:
    part = PER_DATASET_METRICS_DF[PER_DATASET_METRICS_DF['dataset'].astype(str).eq(dataset_name)].copy()
    if part.empty:
        raise ValueError(f'No rows found for dataset {dataset_name!r}')
    part = part.sort_values('method')

    header = ' & '.join(label for label, _, _ in LATEX_METRIC_SPECS) + r' \\'
    rows = []
    for _, row in part.iterrows():
        values = [latex_metric_value(row[col], fmt) for _, col, fmt in LATEX_METRIC_SPECS]
        rows.append('  ' + ' & '.join(values) + r' \\')

    dataset_title = DATASET_DISPLAY.get(dataset_name, dataset_name.replace('_', ' ').title())
    dataset_label = LATEX_DATASET_LABEL.get(dataset_name, dataset_name.replace('_', '-'))
    body = '\n'.join(rows)
    return rf'''
\begin{{table}}[H]
\centering
\scriptsize
\setlength{{\tabcolsep}}{{3.5pt}}
\begin{{tabular}}{{lrrrrrrrr}}
\toprule
{header}
\midrule
{body}
\bottomrule
\end{{tabular}}
\caption{{Per-dataset metrics on {dataset_title}.}}
\label{{tab:per-dataset-metrics-{dataset_label}}}
\end{{table}}
'''.strip()


DATASET_LATEX_TABLES = {dataset: dataset_metrics_to_latex(dataset) for dataset in DATASET_ORDER}
DATASET_LATEX_TABLES_ALL = '\n\n'.join(DATASET_LATEX_TABLES.values())
print(DATASET_LATEX_TABLES_ALL)


## Summary Barplot Confidence Intervals

Compute dataset-level uncertainty bars for the paper summary barplots. Each method contributes one value per dataset; the table reports both t-based 95% confidence intervals and bootstrap percentile intervals across datasets.

In [ ]:
from scipy.stats import t as student_t

SUMMARY_METRIC_SPECS = [
    ('validity', 'Validity (%)', 'validity_ratio', 'higher'),
    ('l1_proximity', 'L1 proximity', 'l1_proximity', 'lower'),
    ('l0_sparsity', 'L0 sparsity', 'l0_sparsity', 'lower'),
    ('robustness', 'Robustness (%)', 'robustness_avg', 'higher'),
    ('interpretability', 'Decision-tree error', 'decision_tree_error', 'lower'),
    ('query_time', 'Query time (s)', 'query_time_s', 'lower'),
    ('plausibility', 'log10(LOF)', 'log10_lof', 'lower'),
    ('privacy_rpr', 'Relative proximity ratio', 'relative_proximity_ratio', 'to_one'),
]

SUMMARY_METHOD_SHORT = {
    'CertCF': 'C',
    'NN': 'N',
    'Growing Spheres': 'G',
    'FACE': 'F',
    'DiCE': 'D',
}
SUMMARY_METHOD_ID = {
    'CertCF': 1,
    'NN': 2,
    'Growing Spheres': 3,
    'FACE': 4,
    'DiCE': 5,
}

BOOTSTRAP_N_RESAMPLES = 20_000
BOOTSTRAP_RANDOM_STATE = 123
CI_LEVEL = 0.95
rng = np.random.default_rng(BOOTSTRAP_RANDOM_STATE)


def mean_ci_from_dataset_values(values: np.ndarray) -> dict[str, float]:
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    n = int(values.size)
    if n == 0:
        return {
            'n_datasets': 0,
            'mean': np.nan,
            'std': np.nan,
            'se': np.nan,
            't_ci_low': np.nan,
            't_ci_high': np.nan,
            't_err_minus': np.nan,
            't_err_plus': np.nan,
            'boot_ci_low': np.nan,
            'boot_ci_high': np.nan,
            'boot_err_minus': np.nan,
            'boot_err_plus': np.nan,
        }

    mean = float(np.mean(values))
    std = float(np.std(values, ddof=1)) if n > 1 else 0.0
    se = std / np.sqrt(n) if n > 1 else 0.0
    if n > 1:
        alpha = 1.0 - CI_LEVEL
        tcrit = float(student_t.ppf(1.0 - alpha / 2.0, df=n - 1))
        t_ci_low = mean - tcrit * se
        t_ci_high = mean + tcrit * se
    else:
        t_ci_low = mean
        t_ci_high = mean

    if n > 1:
        boot_idx = rng.integers(0, n, size=(BOOTSTRAP_N_RESAMPLES, n))
        boot_means = values[boot_idx].mean(axis=1)
        alpha = 1.0 - CI_LEVEL
        boot_ci_low, boot_ci_high = np.quantile(boot_means, [alpha / 2.0, 1.0 - alpha / 2.0])
        boot_ci_low = float(boot_ci_low)
        boot_ci_high = float(boot_ci_high)
    else:
        boot_ci_low = mean
        boot_ci_high = mean

    return {
        'n_datasets': n,
        'mean': mean,
        'std': std,
        'se': float(se),
        't_ci_low': float(t_ci_low),
        't_ci_high': float(t_ci_high),
        't_err_minus': float(mean - t_ci_low),
        't_err_plus': float(t_ci_high - mean),
        'boot_ci_low': boot_ci_low,
        'boot_ci_high': boot_ci_high,
        'boot_err_minus': float(mean - boot_ci_low),
        'boot_err_plus': float(boot_ci_high - mean),
    }


summary_rows = []
for metric_key, metric_label, column, direction in SUMMARY_METRIC_SPECS:
    for method in [method_display(m) for m in METHOD_ORDER]:
        part = PER_DATASET_METRICS_DF.loc[
            PER_DATASET_METRICS_DF['method'].astype(str).eq(str(method)),
            ['dataset', column],
        ].dropna()
        stats = mean_ci_from_dataset_values(part[column].to_numpy(dtype=float))
        summary_rows.append({
            'metric': metric_key,
            'metric_label': metric_label,
            'metric_column': column,
            'direction': direction,
            'method': str(method),
            'method_short': SUMMARY_METHOD_SHORT.get(str(method), str(method)),
            'method_id': SUMMARY_METHOD_ID.get(str(method), np.nan),
            **stats,
        })

SUMMARY_BAR_CI_DF = pd.DataFrame(summary_rows)

# Convenience columns for PGFPlots. For log-scale query-time plots, lower error
# bounds must stay positive.
SUMMARY_BAR_CI_DF['t_ci_low_plot'] = SUMMARY_BAR_CI_DF['t_ci_low']
SUMMARY_BAR_CI_DF['boot_ci_low_plot'] = SUMMARY_BAR_CI_DF['boot_ci_low']
query_mask = SUMMARY_BAR_CI_DF['metric'].eq('query_time')
SUMMARY_BAR_CI_DF.loc[query_mask, 't_ci_low_plot'] = SUMMARY_BAR_CI_DF.loc[query_mask, 't_ci_low_plot'].clip(lower=1e-12)
SUMMARY_BAR_CI_DF.loc[query_mask, 'boot_ci_low_plot'] = SUMMARY_BAR_CI_DF.loc[query_mask, 'boot_ci_low_plot'].clip(lower=1e-12)
SUMMARY_BAR_CI_DF['t_err_minus_plot'] = SUMMARY_BAR_CI_DF['mean'] - SUMMARY_BAR_CI_DF['t_ci_low_plot']
SUMMARY_BAR_CI_DF['boot_err_minus_plot'] = SUMMARY_BAR_CI_DF['mean'] - SUMMARY_BAR_CI_DF['boot_ci_low_plot']
SUMMARY_BAR_CI_DF['t_err_plus_plot'] = SUMMARY_BAR_CI_DF['t_err_plus']
SUMMARY_BAR_CI_DF['boot_err_plus_plot'] = SUMMARY_BAR_CI_DF['boot_err_plus']

print(SUMMARY_BAR_CI_DF.shape)
display(SUMMARY_BAR_CI_DF.round({
    'mean': 4,
    'std': 4,
    'se': 4,
    't_ci_low': 4,
    't_ci_high': 4,
    'boot_ci_low': 4,
    'boot_ci_high': 4,
    't_err_minus_plot': 4,
    't_err_plus_plot': 4,
    'boot_err_minus_plot': 4,
    'boot_err_plus_plot': 4,
}))


## Optional Export

In [ ]:
if EXPORT_TABLES:
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    PER_DATASET_METRICS_DF.to_csv(EXPORT_DIR / 'per_dataset_metrics.csv', index=False)
    if 'SUMMARY_BAR_CI_DF' in globals():
        SUMMARY_BAR_CI_DF.to_csv(EXPORT_DIR / 'summary_metric_bar_cis.csv', index=False)
    PER_DATASET_METRICS_DF.to_latex(EXPORT_DIR / 'per_dataset_metrics.tex', index=False, float_format='%.4f', escape=False)
    for method, table in METHOD_TABLES.items():
        safe_method = method.lower().replace(' ', '_')
        table.to_csv(EXPORT_DIR / f'per_dataset_metrics_{safe_method}.csv', index=False)
        table.to_latex(EXPORT_DIR / f'per_dataset_metrics_{safe_method}.tex', index=False, float_format='%.4f', escape=False)
    if 'DATASET_LATEX_TABLES_ALL' in globals():
        (EXPORT_DIR / 'per_dataset_metrics_by_dataset.tex').write_text(DATASET_LATEX_TABLES_ALL + '\n')
        for dataset_name, latex_table in DATASET_LATEX_TABLES.items():
            safe_dataset = dataset_name.replace('_', '-')
            (EXPORT_DIR / f'per_dataset_metrics_{safe_dataset}.tex').write_text(latex_table + '\n')
    print(f'Exported tables to {EXPORT_DIR}')
else:
    print('EXPORT_TABLES is False; no files written.')
